## Install Dependencies

In [ ]:
!pip install --upgrade langchain langchain-google-genai faiss-cpu sentence-transformers google-colab langchain-community

## 1. Set Up Google Cloud API Key in Colab
This setup authenticates access to Gemini 2.0 Flash via LangChain’s Google GenAI integration.Add the API KEY in the secrets folder of Google Colab.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Retrieve your Google API key stored as Colab secret/environment variable
GOOGLE_API_KEY = API_KEY
genai.configure(api_key=GOOGLE_API_KEY)

## 2. Embed Document Text Directly in the Code

In [ ]:
from langchain.docstore.document import Document

#and Micro Electronics Research Lab
doc_text = """
Elon Musk is a technology entrepreneur and engineer known for founding SpaceX and Tesla.
He was born on June 28, 1971, in Pretoria, South Africa.
His major achievements include advancing space exploration and electric vehicles.
Musk is also involved with Neuralink and The Boring Company.
This document provides a brief overview of Musk's background and accomplishments.
"""

document = Document(page_content=doc_text, metadata={"source": "in-memory-doc"})

## 3. Split Document into Chunks for Better Retrieval
Splitting ensures manageable chunks for embedding and avoids truncation in retrieval.

- The RecursiveCharacterTextSplitter function splits a large text into smaller chunks of a specified size by recursively trying to split on different separators (like paragraphs, lines, spaces) until suitable chunk sizes are obtained, allowing effective processing of long documents.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_split = text_splitter.split_documents([document])

## 4. Use Sentence Transformers for Local Embeddings
This model runs entirely locally, requiring no external API calls, suitable for embedding text chunks and queries.

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

## 5. Create FAISS Vector Store for Efficient Similarity Search
FAISS provides rapid approximate nearest neighbor search over vectors.

In [ ]:
from langchain.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs_split, embeddings)

## 6. Initialize Gemini 2.0 Flash LLM Using the API Key
This loads the powerful Gemini 2.0 Flash model for text generation.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.0,
    max_tokens=None,
    api_key=GOOGLE_API_KEY,
)

## 7. Build the RetrievalQA Chain
LangChain’s RetrievalQA combines retrieval and generation seamlessly.

In [ ]:
from langchain.chains import RetrievalQA

retriever = vectorstore.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,  # to get source documents with answers
    chain_type="stuff",            # concatenates retrieved docs into prompt
)

## 8. Query the RAG System and Print Results
The model returns a grounded answer, supported by the relevant document chunks.

In [ ]:
query = "Who is Elon Musk and what are his major achievements?"

result = qa_chain.invoke({"query": query})

print("Answer:", result['result'])
print("\nSource Documents:")
for doc in result['source_documents']:
    print(f"- {doc.page_content}")